# ML-07 - Baseline Action Score and Top-10 Review

This notebook locks my provisional lane as **Lane 2: Refresh / Content Opportunity Scoring** and builds one transparent baseline rule for a ranked action queue.

The rule is deliberately simple: find visible pages with useful search position but unusually weak CTR, then rank the strongest review candidates first. This is not a model; it is the baseline my Week-5 model must beat.

## 1. Check two signals first

**Plain-language rule idea:** A page deserves CTR/title-meta review when it has enough impressions to matter, ranks close enough that searchers can see it, and still earns very low CTR.

I will check two signals before scoring anything:

1. **CTR vs position** - tied to FlyRank's CTR-fix logic. If lower position tiers have lower CTR, then position context matters before calling CTR weak.
2. **Volume / impressions** - tied to quick-win thinking. If higher impression buckets carry more clicks/sessions, then volume is a reasonable guardrail for review priority.

The starter proxy label (`trend_direction == "down"`) is used only as a review/evaluation receipt below. It is **not** used in the baseline score.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
DATA_PATH = REPO_ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
OUT_DIR = REPO_ROOT / "work" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df["is_down_proxy"] = df["trend_direction"].str.lower().eq("down")

print(f"Loaded {len(df):,} rows from {DATA_PATH.relative_to(REPO_ROOT)}")
print(f"Unique pseudonymized clients: {df['client_id'].nunique()}")
print("Reminder: ctr is a x100 percentage, so ctr=0.50 means 0.50%, not 50%.")

# Signal 1: CTR by position tier, among visible pages.
visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0)].copy()
position_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
ctr_position_table = (
    visible.groupby("position_tier")
    .agg(
        n=("content_id", "size"),
        median_position=("avg_position", "median"),
        mean_ctr_pct=("ctr", "mean"),
        median_ctr_pct=("ctr", "median"),
        down_rate_proxy_pct=("is_down_proxy", lambda s: s.mean() * 100),
    )
    .reindex(position_order)
    .round(2)
)

print("\nSignal 1: CTR vs position tier, visible pages only (impressions_90d >= 500)")
display(ctr_position_table)
print("Verdict: MIXED")
print(
    "CTR generally weakens in worse position tiers, especially past the top 20, "
    "but top_3 and page_1 are close. The rule should use position context and a strict low-CTR cutoff."
)

# Signal 2: volume buckets.
impression_order = ["low", "moderate", "good", "excellent"]
volume_table = (
    df.groupby("impression_tier")
    .agg(
        n=("content_id", "size"),
        median_impressions=("impressions_90d", "median"),
        median_clicks=("clicks_90d", "median"),
        pct_with_clicks=("clicks_90d", lambda s: (s > 0).mean() * 100),
        median_sessions=("sessions_90d", "median"),
        down_rate_proxy_pct=("is_down_proxy", lambda s: s.mean() * 100),
    )
    .reindex(impression_order)
    .round(2)
)

print("\nSignal 2: volume buckets")
display(volume_table)
print("Verdict: CONFIRMED")
print(
    "Higher impression buckets have much higher click/session medians, so an impression floor is a practical way "
    "to avoid spending review time on tiny pages."
)

Loaded 30,000 rows from data\raw\content_refresh_anonymized.csv
Unique pseudonymized clients: 32
Reminder: ctr is a x100 percentage, so ctr=0.50 means 0.50%, not 50%.

Signal 1: CTR vs position tier, visible pages only (impressions_90d >= 500)


,n,median_position,mean_ctr_pct,median_ctr_pct,down_rate_proxy_pct
position_tier,,,,,
top_3,458,2.40,0.35,0.20,74.02
page_1,7064,6.50,0.34,0.24,57.91
striking,4485,13.90,0.27,0.17,61.49
page_3_5,4330,28.25,0.14,0.09,61.36
deep,389,58.30,0.04,0.00,29.82


Verdict: MIXED
CTR generally weakens in worse position tiers, especially past the top 20, but top_3 and page_1 are close. The rule should use position context and a strict low-CTR cutoff.

Signal 2: volume buckets


,n,median_impressions,median_clicks,pct_with_clicks,median_sessions,down_rate_proxy_pct
impression_tier,,,,,,
low,11248,31.0,0.0,15.84,2.0,45.39
moderate,10469,998.0,1.0,66.02,8.0,61.47
good,7205,7249.0,16.0,97.50,41.0,58.61
excellent,1078,48675.0,116.0,99.91,184.0,46.20


Verdict: CONFIRMED
Higher impression buckets have much higher click/session medians, so an impression floor is a practical way to avoid spending review time on tiny pages.


## 2. Build the ranked queue

**One rule:** `visible_low_ctr_page`

A page matches when all of these are true:

- `impressions_90d >= 500`
- `1 <= avg_position <= 20`
- `ctr < 0.50`

**Score:** combine the CTR gap, position opportunity, and search volume. The rule uses only pre-decision observed signals: `impressions_90d`, `avg_position`, and `ctr`.

**Action label:** `review_title_meta` for matches, otherwise `monitor`.

The notebook writes `work/outputs/baseline_action_score.csv`. That CSV stays out of git by design, so I also write `work/outputs/baseline_action_score_metrics.json` as a small committed receipt.

In [2]:
queue = df.copy()

queue["valid_position"] = queue["avg_position"].gt(0)
queue["visible"] = queue["impressions_90d"].ge(500)
queue["reachable_position"] = queue["avg_position"].between(1, 20, inclusive="both")
queue["low_ctr"] = queue["ctr"].lt(0.50)
queue["rule_match"] = queue["visible"] & queue["reachable_position"] & queue["low_ctr"]

position_factor = (21 - queue["avg_position"]).clip(lower=1, upper=20) / 20
ctr_gap_factor = (0.50 - queue["ctr"]).clip(lower=0) / 0.50
volume_factor = np.log1p(queue["impressions_90d"]) / np.log1p(queue["impressions_90d"].max())

queue["baseline_action_score"] = (
    queue["rule_match"].astype(int) * 100 * position_factor * ctr_gap_factor * volume_factor
).round(2)
queue["reason_code"] = np.where(queue["rule_match"], "visible_low_ctr_page", "")
queue["action_label"] = np.where(queue["rule_match"], "review_title_meta", "monitor")

ranked = (
    queue.sort_values(
        ["baseline_action_score", "impressions_90d", "avg_position"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)
ranked.insert(0, "rank", np.arange(1, len(ranked) + 1))

output_columns = [
    "rank",
    "content_id",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "position_tier",
    "sessions_90d",
    "engagement_rate",
    "scroll_rate",
]

csv_path = OUT_DIR / "baseline_action_score.csv"
json_path = OUT_DIR / "baseline_action_score_metrics.json"
ranked[output_columns].to_csv(csv_path, index=False)

base_rate = float(queue["is_down_proxy"].mean())
metrics = {
    "rule_name": "visible_low_ctr_page",
    "lane": "Refresh / Content Opportunity Scoring",
    "rows_scored": int(len(ranked)),
    "rule_matches": int(queue["rule_match"].sum()),
    "score_inputs": ["impressions_90d", "avg_position", "ctr"],
    "reason_code": "visible_low_ctr_page",
    "action_label_for_matches": "review_title_meta",
    "csv_written": "work/outputs/baseline_action_score.csv",
    "base_down_rate_proxy": round(base_rate, 4),
    "proxy_precision_at_10": round(float(ranked.head(10)["is_down_proxy"].mean()), 4),
    "proxy_precision_at_20": round(float(ranked.head(20)["is_down_proxy"].mean()), 4),
    "proxy_precision_at_50": round(float(ranked.head(50)["is_down_proxy"].mean()), 4),
    "leakage_excluded_from_score": [
        "trend_direction",
        "trend_pct",
        "impressions_last_30d",
        "clicks_last_30d",
        "sessions_last_30d",
        "impressions_prev_30d",
        "clicks_prev_30d",
        "sessions_prev_30d",
    ],
}
json_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print(f"Rule matches: {metrics['rule_matches']:,} of {metrics['rows_scored']:,} rows")
print(f"Wrote CSV: {csv_path.relative_to(REPO_ROOT)}")
print(f"Wrote metrics JSON: {json_path.relative_to(REPO_ROOT)}")
print(
    "Proxy precision using trend_direction only as a receipt: "
    f"P@10={metrics['proxy_precision_at_10']:.2f}, "
    f"P@20={metrics['proxy_precision_at_20']:.2f}, "
    f"P@50={metrics['proxy_precision_at_50']:.2f}; "
    f"base down rate={metrics['base_down_rate_proxy']:.2f}"
)

display(ranked[output_columns].head(10))

Rule matches: 9,745 of 30,000 rows


Wrote CSV: work\outputs\baseline_action_score.csv
Wrote metrics JSON: work\outputs\baseline_action_score_metrics.json
Proxy precision using trend_direction only as a receipt: P@10=0.50, P@20=0.65, P@50=0.68; base down rate=0.54


,rank,content_id,baseline_action_score,reason_code,action_label,impressions_90d,clicks_90d,ctr,avg_position,position_tier,sessions_90d,engagement_rate,scroll_rate
0,1,content_8451fc6f034d,83.59,visible_low_ctr_page,review_title_meta,272144,75,0.03,2.3,top_3,99,2.02,2.26
1,2,content_4a6607efcb46,82.34,visible_low_ctr_page,review_title_meta,128068,17,0.01,2.2,top_3,87,2.30,6.11
2,3,content_e12868d1f396,70.49,visible_low_ctr_page,review_title_meta,149712,104,0.07,2.9,top_3,101,5.94,11.97
3,4,content_339b357d04c7,69.30,visible_low_ctr_page,review_title_meta,46879,7,0.01,3.7,page_1,14,0.00,14.29
4,5,content_0022a6b4290f,66.65,visible_low_ctr_page,review_title_meta,29747,22,0.07,1.2,top_3,27,0.00,10.71
5,6,content_954cc45bd437,65.75,visible_low_ctr_page,review_title_meta,15439,6,0.04,1.5,top_3,6,0.00,0.00
6,7,content_f4e210ee0c27,65.64,visible_low_ctr_page,review_title_meta,24784,16,0.06,1.6,top_3,11,0.00,20.00
7,8,content_cbdf5a78dcd0,65.17,visible_low_ctr_page,review_title_meta,14830,3,0.02,2.4,top_3,4,0.00,0.00
8,9,content_fea6a0d13b4a,64.94,visible_low_ctr_page,review_title_meta,79965,56,0.07,3.4,page_1,58,0.00,6.06
9,10,content_8c19996aa890,64.67,visible_low_ctr_page,review_title_meta,509252,785,0.15,2.5,top_3,571,11.73,16.03


## 3. Top-10 review

For each top-ranked row, I read the action, why the rule put it there, and what would make the recommendation wrong. The IDs are pseudonymous; no URLs, titles, queries, or client names are used.

In [3]:
def what_could_make_wrong(row):
    risks = []
    if row["sessions_90d"] < 10:
        risks.append("low sessions may make the business value weak")
    if row["clicks_90d"] >= 50:
        risks.append("it already earns clicks, so low CTR may be normal for this query mix")
    if row["avg_position"] > 10:
        risks.append("position may explain weak CTR more than title/meta quality")
    if not risks:
        risks.append("SERP features, brand intent, or query intent could make low CTR non-actionable")
    return "; ".join(risks)

review_rows = []
for _, row in ranked.head(10).iterrows():
    review_rows.append(
        {
            "rank": int(row["rank"]),
            "content_id": row["content_id"],
            "action": row["action_label"],
            "why_it_is_here": (
                f"{row['impressions_90d']:,} impressions, avg position {row['avg_position']:.1f}, "
                f"CTR {row['ctr']:.2f}%, reason={row['reason_code']}"
            ),
            "what_would_make_it_wrong": what_could_make_wrong(row),
        }
    )

top10_review = pd.DataFrame(review_rows)
display(top10_review)

,rank,content_id,action,why_it_is_here,what_would_make_it_wrong
0,1,content_8451fc6f034d,review_title_meta,"272,144 impressions, avg position 2.3, CTR 0.0...","it already earns clicks, so low CTR may be nor..."
1,2,content_4a6607efcb46,review_title_meta,"128,068 impressions, avg position 2.2, CTR 0.0...","SERP features, brand intent, or query intent c..."
2,3,content_e12868d1f396,review_title_meta,"149,712 impressions, avg position 2.9, CTR 0.0...","it already earns clicks, so low CTR may be nor..."
3,4,content_339b357d04c7,review_title_meta,"46,879 impressions, avg position 3.7, CTR 0.01...","SERP features, brand intent, or query intent c..."
4,5,content_0022a6b4290f,review_title_meta,"29,747 impressions, avg position 1.2, CTR 0.07...","SERP features, brand intent, or query intent c..."
5,6,content_954cc45bd437,review_title_meta,"15,439 impressions, avg position 1.5, CTR 0.04...",low sessions may make the business value weak
6,7,content_f4e210ee0c27,review_title_meta,"24,784 impressions, avg position 1.6, CTR 0.06...","SERP features, brand intent, or query intent c..."
7,8,content_cbdf5a78dcd0,review_title_meta,"14,830 impressions, avg position 2.4, CTR 0.02...",low sessions may make the business value weak
8,9,content_fea6a0d13b4a,review_title_meta,"79,965 impressions, avg position 3.4, CTR 0.07...","it already earns clicks, so low CTR may be nor..."
9,10,content_8c19996aa890,review_title_meta,"509,252 impressions, avg position 2.5, CTR 0.1...","it already earns clicks, so low CTR may be nor..."


## 4. Weak picks and leakage check

A useful baseline should expose its own weak spots. The rule is intentionally narrow: it prioritizes visible low-CTR pages, but it does not know the raw query, the SERP layout, the business value of the page, or whether a title/meta edit is actually possible.

In [4]:
weak = ranked.head(20).copy()
weak_notes = []
for _, row in weak.iterrows():
    notes = []
    if row["sessions_90d"] < 10:
        notes.append("low sessions")
    if row["clicks_90d"] == 0:
        notes.append("zero clicks")
    if row["avg_position"] > 15:
        notes.append("near bottom of top-20")
    if row["engagement_rate"] < 10 and row["sessions_90d"] >= 10:
        notes.append("weak engagement after click")
    weak_notes.append("; ".join(notes) if notes else "looks reasonable for manual review")

weak["skeptic_note"] = weak_notes
weak_review = weak[[
    "rank",
    "content_id",
    "baseline_action_score",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "sessions_90d",
    "engagement_rate",
    "skeptic_note",
]]

display(weak_review)

score_inputs = ["impressions_90d", "avg_position", "ctr"]
blocked_inputs = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
]

print("Leakage check")
print("Score inputs used:", score_inputs)
print("Excluded from score:", blocked_inputs)
print("No product flags, future-window fields, trend_direction, or trend_pct are used in the baseline score.")

,rank,content_id,baseline_action_score,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,engagement_rate,skeptic_note
0,1,content_8451fc6f034d,83.59,272144,75,0.03,2.3,99,2.02,weak engagement after click
1,2,content_4a6607efcb46,82.34,128068,17,0.01,2.2,87,2.30,weak engagement after click
2,3,content_e12868d1f396,70.49,149712,104,0.07,2.9,101,5.94,weak engagement after click
3,4,content_339b357d04c7,69.30,46879,7,0.01,3.7,14,0.00,weak engagement after click
4,5,content_0022a6b4290f,66.65,29747,22,0.07,1.2,27,0.00,weak engagement after click
5,6,content_954cc45bd437,65.75,15439,6,0.04,1.5,6,0.00,low sessions
6,7,content_f4e210ee0c27,65.64,24784,16,0.06,1.6,11,0.00,weak engagement after click
7,8,content_cbdf5a78dcd0,65.17,14830,3,0.02,2.4,4,0.00,low sessions
8,9,content_fea6a0d13b4a,64.94,79965,56,0.07,3.4,58,0.00,weak engagement after click
9,10,content_8c19996aa890,64.67,509252,785,0.15,2.5,571,11.73,looks reasonable for manual review


Leakage check
Score inputs used: ['impressions_90d', 'avg_position', 'ctr']
Excluded from score: ['trend_direction', 'trend_pct', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']
No product flags, future-window fields, trend_direction, or trend_pct are used in the baseline score.


## Self-check

- [x] Two signal checks are shown with bucket tables and `n`.
- [x] At least one signal is linked to a real FlyRank flag logic: CTR-vs-position and low CTR review.
- [x] Each signal has a one-word verdict: `MIXED` and `CONFIRMED`.
- [x] One rule is encoded with a score, one reason code, and an action label.
- [x] The notebook writes `work/outputs/baseline_action_score.csv`.
- [x] The top-10 rows are reviewed with action, reason, and what would make each wrong.
- [x] The score uses no future-window or label-derived inputs.
- [x] The metrics receipt is written to `work/outputs/baseline_action_score_metrics.json` for commit.